# CreditFlow — Train pipeline trên Colab GPU T4

Chạy **100% trên Colab**, không train ở máy local. Repo được clone vào `/content/CreditFlow`.

**Pipeline (`scripts/train_models.py`):** `write_dataset(n=5000)` → fit LogReg/XGBoost (+ các model trong factory) → chọn **lowest business cost** (FN cost 5 >> FP cost 1, threshold tối ưu trên validation) → ghi `models/production/pipeline.joblib` (+ `benchmark_results.csv/.json`, `meta.json`, `reference_stats.json`).

> Runtime: **Colab → Change runtime type → T4 GPU**. Chạy từng cell theo thứ tự.

In [1]:
# Cell 1 — Check GPU T4
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('[info] CPU-only cũng train được (tabular sklearn/XGBoost) — có GPU càng tốt.')

Thu Sep 17 19:14:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — Clone repo (link thật từ `git remote -v` của máy local)
REPO_URL = 'https://github.com/imtarget05/CreditFlow.git'
!rm -rf /content/CreditFlow
!git clone $REPO_URL /content/CreditFlow
!ls /content/CreditFlow
!ls /content/CreditFlow/scripts /content/CreditFlow/pipeline/data

Cloning into '/content/CreditFlow'...
remote: Enumerating objects: 506, done.
remote: Counting objects: 100% (506/506), done.
remote: Compressing objects: 100% (306/306), done.
remote: Total 506 (delta 215), reused 460 (delta 169), pack-reused 0 (from 0)
Receiving objects: 100% (506/506), 730.32 KiB | 15.88 MiB/s, done.
Resolving deltas: 100% (215/215), done.
backend		    docs       pipeline		     requirements.txt
data		    frontend   README.md	     scripts
deploy		    models     render.yaml	     SECURITY.md
docker-compose.yml  notebooks  requirements-dev.txt  tests
/content/CreditFlow/pipeline/data:
generate_dataset.py  ingest.py	__init__.py  mapping.yaml

/content/CreditFlow/scripts:
check_render_api.py  eval_explanations.py  train_models.py  verify_deploy.py


In [3]:
# Cell 3 — Cài đặt (chỉ trên Colab)
# 1. Khôi phục lại trạng thái thư viện chuẩn của Colab tránh bị lỗi xung đột
!pip install -q --force-reinstall numpy pandas scipy scikit-learn xgboost

# 2. Cài đặt các thư viện nhẹ khác từ requirements.txt
!grep -vE "numpy|pandas|scikit-learn|xgboost|scipy" /content/CreditFlow/requirements.txt > /tmp/filtered_requirements.txt
%pip install -q -r /tmp/filtered_requirements.txt

# 3. Import kiểm tra
import sklearn, xgboost, pandas, numpy, joblib
print('sklearn:', sklearn.__version__)
print('xgboost:', xgboost.__version__)
print('pandas:', pandas.__version__)
print('numpy:', numpy.__version__)

print('\n[LƯU Ý]: Nếu vẫn gặp lỗi AttributeError, hãy nhấn menu: "Runtime" -> "Restart Session" (hoặc Ctrl+M .) rồi chạy lại từ đầu để hệ thống áp dụng!')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.
sklearn: 1.9.1
xgboost: 3.4.1
pandas: 3.0.5
numpy: 2.5.3

[LƯU Ý]: Nếu vẫn gặp lỗi AttributeError, hãy nhấn menu: "Runtime" -> "Restart Session" (hoặc Ctrl+M .) rồi chạy lại từ đầu để hệ thống áp dụng!


In [9]:
# Cell 4 — Mount Google Drive để backup/tải artifact
from google.colab import drive
try:
    drive.mount('/content/drive')
    print('[ok] Mount Google Drive thành công!')
except Exception as e:
    print('[!] Có lỗi xảy ra khi mount Drive:', e)

Mounted at /content/drive
[ok] Mount Google Drive thành công!


In [5]:
# Cell 5 — Train: write_dataset(5000) + fit LogReg/XGBoost + chọn lowest cost
# Script tự sinh data nếu thiếu (data/creditflow_dataset.csv, n=5000, seed=42)
# và ghi models/production/pipeline.joblib + benchmark + meta.json.
!cd /content/CreditFlow && python scripts/train_models.py --reason "colab T4 retrain"
# Buộc sinh lại dataset (nếu muốn):
# !cd /content/CreditFlow && CREDITFLOW_REGENERATE=1 python scripts/train_models.py --reason "colab T4 retrain (regenerated)"


[mlflow] not available, skipping tracking (No module named 'mlflow')
              model  best_threshold  business_cost  val_accuracy  val_precision  val_recall  val_f1  val_roc_auc  test_accuracy  test_precision  test_recall  test_f1  test_roc_auc
logistic_regression            0.20          201.0        0.8387         0.4420        0.80  0.5694       0.8817         0.8373          0.4321         0.70   0.5344        0.8672
      random_forest            0.25          238.0        0.8427         0.4430        0.70  0.5426       0.8578         0.8493          0.4526         0.62   0.5232        0.8472
            xgboost            0.05          273.0        0.7853         0.3512        0.72  0.4721       0.8418         0.7773          0.3350         0.68   0.4488        0.8233
      decision_tree            0.05          364.0        0.8133         0.3438        0.44  0.3860       0.6554         0.8400          0.4020         0.41   0.4059        0.6581

=== PRODUCTION MODEL ===
{
  

In [6]:
# Cell 6 — Kiểm tra artifact + backup ra Drive (nếu đã mount)
import os
PROD = '/content/CreditFlow/models/production'
for p in ['pipeline.joblib', 'benchmark_results.csv', 'benchmark_results.json', 'meta.json', 'reference_stats.json']:
    fp = os.path.join(PROD, p)
    print(('OK  ' if os.path.exists(fp) else 'MISS'), fp)
print()
!ls -lh /content/CreditFlow/models/production/ /content/CreditFlow/data/ 2>&1
print()
# Xem model nào thắng (lowest business cost):
!cat /content/CreditFlow/models/production/meta.json

# Backup ra Drive (chỉ chạy khi đã mount ở Cell 4)
import os
if os.path.isdir('/content/drive/MyDrive'):
    !mkdir -p /content/drive/MyDrive/CreditFlow
    !cp -v /content/CreditFlow/models/production/pipeline.joblib /content/CreditFlow/models/production/meta.json /content/CreditFlow/models/production/benchmark_results.csv /content/drive/MyDrive/CreditFlow/
    print('[ok] đã copy artifact ra /content/drive/MyDrive/CreditFlow/')
else:
    print('[info] Drive chưa mount — tải trực tiếp ở Cell 7.')

OK   /content/CreditFlow/models/production/pipeline.joblib
OK   /content/CreditFlow/models/production/benchmark_results.csv
OK   /content/CreditFlow/models/production/benchmark_results.json
OK   /content/CreditFlow/models/production/meta.json
OK   /content/CreditFlow/models/production/reference_stats.json

/content/CreditFlow/data/:
total 496K
-rw-r--r-- 1 root root 482K Sep 17 19:14 creditflow_dataset.csv
drwxr-xr-x 2 root root 4.0K Sep 17 19:14 metadata
drwxr-xr-x 2 root root 4.0K Sep 17 19:14 processed
drwxr-xr-x 2 root root 4.0K Sep 17 19:14 raw

/content/CreditFlow/models/production/:
total 24K
-rw-r--r-- 1 root root  507 Sep 17 19:15 benchmark_results.csv
-rw-r--r-- 1 root root 1.4K Sep 17 19:15 benchmark_results.json
-rw-r--r-- 1 root root 1.2K Sep 17 19:15 meta.json
-rw-r--r-- 1 root root 3.2K Sep 17 19:15 pipeline.joblib
-rw-r--r-- 1 root root 7.2K Sep 17 19:15 reference_stats.json

{
  "model_name": "logistic_regression",
  "version": "logistic_regression_v001",
  "trained_at

In [7]:
# Cell 7 — Hướng dẫn copy artifact về máy local
print('Pull về máy — cách 1: Files panel (chuột phải > Download) các file:')
print('  /content/CreditFlow/models/production/pipeline.joblib')
print('  /content/CreditFlow/models/production/meta.json')
print('  /content/CreditFlow/models/production/benchmark_results.csv')
print()
print('Cách 2 (code):')
print('  from google.colab import files')
print('  files.download(\'/content/CreditFlow/models/production/pipeline.joblib\')')
print('  files.download(\'/content/CreditFlow/models/production/meta.json\')')
print('  files.download(\'/content/CreditFlow/models/production/benchmark_results.csv\')')
print()
print('Máy local: copy các file vào <repo>/models/production/ (ghi đè artifact cũ).')

Pull về máy — cách 1: Files panel (chuột phải > Download) các file:
  /content/CreditFlow/models/production/pipeline.joblib
  /content/CreditFlow/models/production/meta.json
  /content/CreditFlow/models/production/benchmark_results.csv

Cách 2 (code):
  from google.colab import files
  files.download('/content/CreditFlow/models/production/pipeline.joblib')
  files.download('/content/CreditFlow/models/production/meta.json')
  files.download('/content/CreditFlow/models/production/benchmark_results.csv')

Máy local: copy các file vào <repo>/models/production/ (ghi đè artifact cũ).


In [8]:
# Cell 8 — Tải trực tiếp tất cả artifacts về máy local
from google.colab import files
import os

artifacts = [
    '/content/CreditFlow/models/production/pipeline.joblib',
    '/content/CreditFlow/models/production/meta.json',
    '/content/CreditFlow/models/production/benchmark_results.csv'
]

print("Đang bắt đầu tải xuống các files...")
for filepath in artifacts:
    if os.path.exists(filepath):
        print(f"Tải xuống: {filepath}")
        files.download(filepath)
    else:
        print(f"[Lỗi] Không tìm thấy file: {filepath}")

Đang bắt đầu tải xuống các files...
Tải xuống: /content/CreditFlow/models/production/pipeline.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tải xuống: /content/CreditFlow/models/production/meta.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tải xuống: /content/CreditFlow/models/production/benchmark_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>